In [1]:
# ETL (Extract - Transform - Load)
import pandas as pd
import mysql.connector
from mysql.connector import Error

In [2]:
#None devuelve lista de dataframes
# xlrd es la maquina para xls, para xlss es otro
datos = pd.read_excel('calificaciones.xls', sheet_name=None, engine='xlrd', keep_default_na=False)

In [3]:
hojas = list(datos.keys())
hojas

['9D', '9B']

In [4]:
arreglo = []
for i, nombre in enumerate(hojas, 1):
    df = datos[nombre]
    df['Grupo'] = nombre
    arreglo.append(df)

df_consolidado = pd.concat(arreglo)
df_consolidado 

,#,Nombre,Asistencia,T1,T2,Examen,Total,SABER,HACER,SER,Ordinario,Recuperación,Grupo
0,1,1,75,100,80,64,"72,9",NA,NA,NA,NA,,9D
1,2,2,100,100,100,82,"89,2",DE,DE,AU,DE,,9D
2,3,3,100,100,100,76,"85,6",DE,DE,AU,DE,,9D
3,4,4,100,100,80,75,"82,0",SA,SA,AU,SA,,9D
4,5,5,100,100,100,85,"91,0",DE,DE,AU,DE,,9D
5,6,6,100,100,100,57,"74,2",NA,NA,AU,NA,,9D
6,7,7,100,100,80,84,"87,4",DE,DE,AU,DE,,9D
7,8,8,75,100,100,70,"79,5",NA,NA,NA,NA,SA,9D
8,9,9,100,100,100,72,"83,2",SA,SA,AU,SA,,9D
9,10,10,100,100,80,70,"79,0",NA,NA,AU,NA,SA,9D


In [5]:
df_consolidado['Recuperación'] = df_consolidado['Recuperación'].replace('', pd.NA) # Le pone un vacio de pandas
# Formas de eliminar vacios: eliminar vacios, o sistituir con media, mediana o moda, analizar lo que se 
# estan haciendo para saber que hacer
# en este caso se quedará con los vacios
df_consolidado

,#,Nombre,Asistencia,T1,T2,Examen,Total,SABER,HACER,SER,Ordinario,Recuperación,Grupo
0,1,1,75,100,80,64,"72,9",NA,NA,NA,NA,NaN,9D
1,2,2,100,100,100,82,"89,2",DE,DE,AU,DE,NaN,9D
2,3,3,100,100,100,76,"85,6",DE,DE,AU,DE,NaN,9D
3,4,4,100,100,80,75,"82,0",SA,SA,AU,SA,NaN,9D
4,5,5,100,100,100,85,"91,0",DE,DE,AU,DE,NaN,9D
5,6,6,100,100,100,57,"74,2",NA,NA,AU,NA,NaN,9D
6,7,7,100,100,80,84,"87,4",DE,DE,AU,DE,NaN,9D
7,8,8,75,100,100,70,"79,5",NA,NA,NA,NA,SA,9D
8,9,9,100,100,100,72,"83,2",SA,SA,AU,SA,NaN,9D
9,10,10,100,100,80,70,"79,0",NA,NA,AU,NA,SA,9D


In [6]:
# Limpiar vacios
# sin el how='all', solo puro (), elimina todos los registros que tengan una sola celda vacia
df_consolidado = df_consolidado.dropna(how='all') # how='all' elimina todos aquellos donde toda la fila este vacia

In [7]:
df_consolidado.duplicated().sum() 

np.int64(0)

In [8]:
# Eliminar los duplicados
df_consolidado = df_consolidado.drop_duplicates() # Crea copia y sobrescribe el dataframe original
# df_consolidado.drop_duplicates(inplace=True) # Borra los duplicados sobre el mismo dataframe original

In [9]:
# Validaciones
df_consolidado['Ordinario'] = df_consolidado['Ordinario'].str.strip() #str -> accesos
df_consolidado['Recuperación'] = df_consolidado['Recuperación'].str.strip()

condicion =(
    (df_consolidado['Ordinario'].isin(['NA', 'SA', 'DE', 'AU'])) 
    &
    (df_consolidado['Recuperación'].isin([pd.NA, 'NA', 'SA'])) 
)
df_consolidado = df_consolidado[condicion]

df_consolidado = df_consolidado[df_consolidado['Grupo'].str.match(r'^\d{1,2}[A-J]$')]
df_consolidado.shape

(51, 13)

In [10]:
# Preparación de los datos
grupo = df_consolidado[['Grupo']].drop_duplicates()
# Rangos de python, investigar, 1 a 12, por que el primero si lo incluye pero el segundo no, es un rango abierto
tipocalificacion = df_consolidado.columns.tolist()[10:12]
alumnos = df_consolidado[['Nombre', 'Grupo']].drop_duplicates() # Se agrega grupo para poderlos identificar
ordinario = df_consolidado[['Nombre', 'Grupo', 'Ordinario']].drop_duplicates()
recuperacion = df_consolidado[['Nombre', 'Grupo', 'Recuperación']].drop_duplicates().dropna()

In [11]:
config_db = {
    'host': 'localhost', 
    'user':'root', 
    'password': 'root',
    'port': 3306,
    'database': 'calificaciones',
    'charset': 'utf8mb4',
    'use_unicode': True
}

try:
    connection = mysql.connector.connect(**config_db)
    if connection.is_connected():
        print('Conexión realizada')
    else:
        print('Error de conexión')
except Error as e:
    print(f"Error al conectar: {e}")

Conexión realizada


In [12]:
# Grupo
cursor = connection.cursor()
for i, nombregrupo in grupo.iterrows():
    cursor.execute('INSERT IGNORE INTO Grupo(nombre) VALUES(%s)', (nombregrupo['Grupo'],))
connection.commit()

In [13]:
# Tipo calificación
cursor = connection.cursor()
for nombretipo in tipocalificacion:
    cursor.execute('INSERT IGNORE INTO TipoCalificacion(nombre) VALUES(%s)', (nombretipo,))
connection.commit()

In [14]:
# Alumno
cursor = connection.cursor()
for i, nombrealumno in alumnos.iterrows():  # Siempre poner la i en el for cuando es un dataframe y el iterrows
    cursor.execute('INSERT IGNORE INTO Alumno(nombre) VALUES(%s)', (nombrealumno['Nombre'],))
    # Recuperar el id del alumno insertado
    alumno_id = 0
    if cursor.lastrowid != 0:
        alumno_id = cursor.lastrowid
    # Buscar el id del Grupo
    cursor.execute('SELECT idGrupo FROM Grupo WHERE nombre = %s', (nombrealumno['Grupo'],))
    result = cursor.fetchone()
    cursor.fetchall() # Par asegurarse que el cursor pueda cerrarse
    grupo_id = result[0]
    # Insertar en AlumnoGrupo
    cursor.execute('INSERT IGNORE INTO AlumnoGrupo(idAlumno, idGrupo) VALUES(%s, %s)', (alumno_id,grupo_id))
connection.commit()

In [15]:
# Calificación
cursor = connection.cursor()
for i, calificacion in ordinario.iterrows():
    # Recuperar el idAlumnoGrupo
    cursor.execute("""SELECT idAlumnoGrupo FROM AlumnoGrupo ag JOIN Alumno a 
        ON ag.idAlumno = a.idAlumno
        JOIN Grupo g ON ag.idGrupo = g.idGrupo
        WHERE a.nombre = %s AND g.nombre = %s;""", (calificacion['Nombre'], calificacion['Grupo']))
    result = cursor.fetchone()
    cursor.fetchall()
    idAlumnoGrupo = result[0]
    
    # Recuperar el idTipoCalificación
    cursor.execute('SELECT idTipoCalificacion FROM TipoCalificacion WHERE nombre = %s', ('Ordinario',))
    result = cursor.fetchone()
    cursor.fetchall()
    idTipoCalificacion = result[0]
    
    # Insertar calificacion
    cursor.execute("""INSERT INTO Calificacion(idAlumnoGrupo, idTipoCalificacion, valor)
    VALUES(%s,%s,%s)""", (idAlumnoGrupo, idTipoCalificacion, calificacion['Ordinario']))
connection.commit()

In [ ]:
# REVISAR EL ULTIMO de recuperacion, que me faltó copiarlo